In [1]:
from zipfile import ZipFile

In [4]:
file_name = "dataset.zip"
with ZipFile(file_name,'r') as zip:
    # extracting all the files
    print('Extracting all the files now...')
    zip.extractall()
    print('Done!')
  
   

Extracting all the files now...
Done!


In [5]:
pip install face_recognition

     |████████████████████████████████| 100.1 MB 8.2 kB/s 
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566185 sha256=e0f24cb5421127bd2a1de0c127cca7f1440f19e3e993858704cf2b20c23fe444
  Stored in directory: /root/.cache/pip/wheels/d6/81/3c/884bcd5e1c120ff548d57c2ecc9ebf3281c9a6f7c0e7e7947a
Successfully built face-recognition-models


In [7]:
from PIL import Image, ImageDraw
import face_recognition
import os
def eye_cropper(folders):
    # Establish count for iterative file saving
    count = 0

    # For loop going through each image file
    for folder in os.listdir(folders):
        for file in os.listdir(folders + '/' + folder):
          
            # Using Facial Recognition Library on Image
            image = face_recognition.load_image_file(folders + '/' + folder + '/' + file)
            # create a variable for the facial feature coordinates
            face_landmarks_list = face_recognition.face_landmarks(image)
          
            # create a placeholder list for the eye coordinates
            eyes = []
            try:
                eyes.append(face_landmarks_list[0]['left_eye'])
                eyes.append(face_landmarks_list[0]['right_eye'])
            except:
                continue
            # establish the max x and y coordinates of the eye
            for eye in eyes:
                x_max = max([coordinate[0] for coordinate in eye])
                x_min = min([coordinate[0] for coordinate in eye])
                y_max = max([coordinate[1] for coordinate in eye])
                y_min = min([coordinate[1] for coordinate in eye])
              # establish the range of x and y coordinates    
                x_range = x_max - x_min
                y_range = y_max - y_min
              
                # to make sure the full eye is captured,
                # calculate the coordinates of a square that has 50%
                # cushion added to the axis with a larger range
                if x_range > y_range:
                    right = round(.5*x_range) + x_max
                    left = x_min - round(.5*x_range)
                    bottom = round(((right-left) - y_range))/2 + y_max
                    top = y_min - round(((right-left) - y_range))/2
                else:
                    bottom = round(.5*y_range) + y_max
                    top = y_min - round(.5*y_range)
                    right = round(((bottom-top) - x_range))/2 + x_max
                    left = x_min - round(((bottom-top) - x_range))/2
              
                #crop original image using the cushioned coordinates
                im = Image.open(folders + '/' + folder + '/' + file)
                im = im.crop((left, top, right, bottom))
                
                # resize image for input into our model
                im = im.resize((80,80))
              
                # save file to output folder
                im.save('images_croppedeyes.png')
              
                # increase count for iterative file saving
                count += 1
                # print count every 200 photos to monitor progress
                if count % 200 == 0:
                    print(count)
    
# Call function to crop full-face eye images
eye_cropper('dataset_new/train')

200
400
600
800
1000
1200
1400
1600
1800


Importing the images and  preprocessing them for the model

In [8]:
import cv2
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from keras.models import Sequential
from keras.layers import Dense,Flatten,Conv2D,MaxPooling2D,Dropout

In [12]:
import numpy as np

In [9]:
def load_images_from_folder(folder, eyes = 0):
    count = 0
    error_count = 0
    images = []
    for filename in os.listdir(folder):
        try:
            img = cv2.imread(os.path.join(folder,filename))
            img = cv2.resize(img, (80,80)) ## Resizing the images
            ## for eyes if it is 0: open, 1: close
            images.append([img, eyes])
        except:
            error_count += 1
            print('ErrorCount = ' + str(error_count))
            continue
        
        count += 1
        if count % 500 == 0:
            print('Succesful Image Import Count = ' + str(count))

    return images

folder="dataset_new/train/Open"
open_eyes = load_images_from_folder(folder, 0)

folder="dataset_new/train/Closed"
closed_eyes = load_images_from_folder(folder, 1)
eyes = closed_eyes + open_eyes

Succesful Image Import Count = 500
Succesful Image Import Count = 500


In [10]:
X = [] 
y = [] 
for features, label in eyes: 
     X.append(features)
     y.append(label)

In [13]:
X = np.array(X).reshape(-1, 80, 80, 3)
y = np.array(y)
X = X/255.0

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y)

In [18]:
# Instantiate the model
model = Sequential()

# Adding first three convolutional layers
model.add(Conv2D(
                filters = 32, # number of filters
                kernel_size = (3,3), # height/width of filter
                activation = 'relu', # activation function 
                input_shape = (80,80,3) # shape of input (image)
                ))
model.add(Conv2D(
                filters = 32, # number of filters
                kernel_size = (3,3), # height/width of filter
                activation = 'relu' # activation function 
                ))
model.add(Conv2D(
                filters = 32, # number of filters
                kernel_size = (3,3), # height/width of filter
                activation = 'relu' # activation function 
                ))


# Adding pooling after convolutional layers
model.add(MaxPooling2D(pool_size = (2,2))) # Dimensions of the region that you are pooling

# Adding second set of convolutional layers
model.add(Conv2D(
                filters = 32, # number of filters
                kernel_size = (3,3), # height/width of filter
                activation = 'relu' # activation function 
                ))
model.add(Conv2D(
                filters = 32, # number of filters
                kernel_size = (3,3), # height/width of filter
                activation = 'relu' # activation function 
                ))


# Add last pooling layer.
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())

# Adding first dense layer with 256 nodes
model.add(Dense(256, activation='relu'))

# Adding a dropout layer to avoid overfitting
model.add(Dropout(0.3))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3)) 

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

# adding output layer
model.add(Dense(1, activation = 'sigmoid'))

# compiling the model
model.compile(loss='binary_crossentropy',
                optimizer='adam',
                metrics=[tf.keras.metrics.AUC(curve = 'PR')])

# fitting the model
model.fit(X_train,
            y_train,
            batch_size=800,
            validation_data=(X_test, y_test),
            epochs=50)

# evaluate the model 
model.evaluate(X_test, y_test, verbose=1)

Epoch 1/50
2/2 [==============================] - 4s 1s/step - loss: 0.6941 - auc_3: 0.5241 - val_loss: 0.6845 - val_auc_3: 0.7319
Epoch 2/50
2/2 [==============================] - 1s 228ms/step - loss: 0.6878 - auc_3: 0.5650 - val_loss: 0.6628 - val_auc_3: 0.9298
Epoch 3/50
2/2 [==============================] - 1s 229ms/step - loss: 0.6661 - auc_3: 0.7141 - val_loss: 0.5885 - val_auc_3: 0.9385
Epoch 4/50
2/2 [==============================] - 1s 225ms/step - loss: 0.5886 - auc_3: 0.8651 - val_loss: 0.5141 - val_auc_3: 0.8954
Epoch 5/50
2/2 [==============================] - 1s 242ms/step - loss: 0.5199 - auc_3: 0.8235 - val_loss: 0.3661 - val_auc_3: 0.9296
Epoch 6/50
2/2 [==============================] - 1s 230ms/step - loss: 0.3791 - auc_3: 0.9221 - val_loss: 0.3391 - val_auc_3: 0.9229
Epoch 7/50
2/2 [==============================] - 1s 225ms/step - loss: 0.3250 - auc_3: 0.9377 - val_loss: 0.3768 - val_auc_3: 0.9335
Epoch 8/50
2/2 [==============================] - 1s 228ms/step -

[0.24191810190677643, 0.9704952239990234]

In [19]:
model.save('best_model.h5')